In [59]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

### Orders

In [60]:
orders = spark.read.csv("output/bronze/orders.csv", header=True, inferSchema=True)

- As Columns `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` are all timestamp values and about the delivery of order, these columns values  are reduqired and not dropping

In [61]:
orders = orders.withColumn("_is_valid", when(col("order_delivered_customer_date") <  col("order_purchase_timestamp"), False) \
           .otherwise(True))


In [62]:
orders = orders.dropDuplicates(['customer_id','order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date', 
 'order_estimated_delivery_date',
 'order_id',
 'order_purchase_timestamp',
 'order_status'])

try:
    orders_silver = spark.read.csv("work/output/silver/orders.csv", header=True, inferSchema=True)
    silver_cleaned = orders_silver.join(orders, on=['order_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(orders)
except Exception:
    final_silver = orders

final_silver.coalesce(1).write.mode("overwrite").csv("output/silver/orders.csv", header=True)

### Order_Items

In [63]:
order_items =spark.read.csv("output/bronze/order_items.csv", header=True, inferSchema=True)

In [64]:
order_items = order_items.withColumn("_is_valid", when((col("price") < 0) | (col("freight_value") < 0), False).when(col("order_id").isNull() | col("product_id").isNull() | col("seller_id").isNull(), False).otherwise(True))
order_items = order_items.withColumn("freight_value", col("freight_value").cast('decimal')).withColumn("price", col("price").cast("decimal"))

In [65]:
order_items = order_items.dropDuplicates(['freight_value', 'order_id', 'order_item_id','price', 'product_id','seller_id','shipping_limit_date'])

try:
    order_items_silver = spark.read.csv("work/output/silver/order_items.csv", header=True, inferSchema=True)
    silver_cleaned = order_items_silver.join(order_items, on=['order_id', 'order_item_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(order_items)
except Exception:
    final_silver = order_items

final_silver.coalesce(1).write.mode("overwrite").csv("output/silver/order_items.csv", header=True)

### Customers

In [66]:
customers =spark.read.csv("output/bronze/customers.csv", header=True, inferSchema=True)

In [67]:
customers = customers.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast('string'))

In [68]:
customers = customers.withColumn("_is_valid", when(col("customer_id").isNull() | col("customer_unique_id").isNull(), False).otherwise(True))

In [69]:
customers = customers.dropDuplicates(['customer_city','customer_id','customer_state','customer_unique_id','customer_zip_code_prefix'])

try:
    customers_silver = spark.read.csv("work/output/silver/order_items.csv", header=True, inferSchema=True)
    silver_cleaned = customers_silver.join(customers, on=['customer_unique_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(customers)
except Exception:
    final_silver = customers

final_silver.coalesce(1).write.mode("overwrite").csv("output/silver/customers.csv", header=True)

### Payments

In [70]:
payments = spark.read.csv("output/bronze/payments.csv", header=True, inferSchema=True)

In [71]:
payments = payments.withColumn("payment_value", col("payment_value").cast('decimal')).withColumn("_is_valid", when(col("payment_value")< 0 , False).when(col("order_id").isNull(), False).otherwise(True)) 

In [72]:
payments = payments.dropDuplicates(['order_id','payment_installments','payment_sequential','payment_type','payment_value'])

try:
    payments_silver = spark.read.csv("work/output/silver/payments.csv", header=True, inferSchema=True)
    silver_cleaned = payments_silver.join(payments, on=['order_id', 'payment_type'], how="left_anti")
    final_silver = silver_cleaned.unionByName(payments)
except Exception:
    final_silver = payments

final_silver.coalesce(1).write.mode("overwrite").csv("output/silver/payments.csv", header=True)

### Products

In [ ]:
products = spark.read.csv("output/bronze/products.csv", inferSchema=True, header=True)

- As columns `product_category_name`, `product_description_length`, `product_name_length`, and `product_photos_qty` having  null records, replacing the null's with `unkonwn` for string and `0` fro int/decimal column's.

In [ ]:
products = products.withColumn("product_category_name", coalesce(col("product_category_name"), lit("UNKNOWN"))) \
.withColumns({"product_description_lenght":coalesce(col("product_description_lenght"), lit(0)),
              "product_name_lenght" :coalesce(col("product_name_lenght"), lit(0)),
              "product_photos_qty": coalesce(col("product_photos_qty"), lit(0))}).withColumn("product_weight_g", col("product_weight_g").cast('decimal')) \
.withColumn("product_width_cm", col("product_width_cm").cast('decimal')) \
.withColumn("_is_valid", when(col("product_id").isNull(), False).otherwise(True))

In [ ]:
products = products.dropDuplicates(['product_category_name',
 'product_description_lenght',
 'product_height_cm',
 'product_id',
 'product_length_cm',
 'product_name_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_width_cm'])

try:
    products_silver = spark.read.csv("work/output/silver/products.csv", header=True, inferSchema=True)
    silver_cleaned = products_silver.join(products, on=['product_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(products)
except Exception:
    final_silver = products

final_silver.coalesce(1).write.mode("overwrite").csv("output/silver/products.csv", header=True)

### sellers

In [ ]:
sellers = spark.read.csv("output/bronze/sellers.csv", header=True, inferSchema=True)

In [ ]:
sellers = sellers.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast('string')).withColumn("_is_valid", when(col("seller_id").isNull(),False).otherwise(True))

In [ ]:
sellers = sellers.dropDuplicates(['seller_city','seller_id','seller_state','seller_zip_code_prefix'])

In [ ]:
try:
    sellers_silver = spark.read.csv("work/output/silver/sellers.csv", header=True, inferSchema=True)
    silver_cleaned = sellers_silver.join(sellers, on=['seller_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(sellers)
except Exception:
    final_silver = sellers

final_silver.coalesce(1).write.mode("overwrite").csv("output/silver/sellers.csv", header=True)